# House Price Prediction — Modelling Pipeline

**Course:** Machine Learning, Concept Integration Assessment 2 (BITS Pilani Dubai)  
**Team:** Nabhya Yadav, MVS Maheedhar Reddy, Nivedita Mahesh, Ponnamanda Hruday Manas

This notebook runs the full pipeline on the Ames Housing data (Kaggle `train.csv`), from setup to the final test-set evaluation. Each section starts with a short note covering **what** the step does, **why** it is needed and **what to look for** in the output.

To rebuild everything in a fresh Colab session, use **Runtime → Run all**.

## 0. Setup

**What:** installs the libraries Colab may lack, imports everything, and downloads `train.csv` into `data/`.

**Why:** Colab sessions are temporary, so every file in `/content/` is deleted when the session ends. Putting all setup at the top means one "Run all" rebuilds the session from nothing. We fix a random seed (`RANDOM_STATE = 42`) so the split and every model give the same results on every run (reproducibility).

**What to look for:** the printed library versions (scikit-learn must be **≥ 1.4** for `TargetEncoder`), and a confirmation that `data/train.csv` exists.

In [12]:
# Install or upgrade the libraries we rely on.
# %pip (not !pip) installs into the kernel this notebook is running on.
# scikit-learn >= 1.4 is required for TargetEncoder (Neighborhood encoding, later).
# -q keeps the install log short.
%pip install -q "scikit-learn>=1.4" "xgboost>=2.0" "shap>=0.44" "gdown>=5.0"

In [13]:
import os

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

# xgboost and shap are only used in later sections. Importing them here
# means a failed install shows up in the first minute, not at the end of the run.
import shap
import xgboost

# One seed for every random step (split, CV folds, models) so results are reproducible.
RANDOM_STATE = 42

print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost     :", xgboost.__version__)
print("shap        :", shap.__version__)

pandas      : 2.2.3
numpy       : 2.4.6
scikit-learn: 1.6.1
xgboost     : 3.4.1
shap        : 0.54.1


In [14]:
DATA_DIR = "data"
DATA_PATH = os.path.join(DATA_DIR, "train.csv")
# Shared Google Drive copy of Kaggle's train.csv (the CSV is not committed to git:
# Kaggle's competition rules restrict redistribution).
DRIVE_FILE_ID = "1kh3Fz_JmR75a22_1mxfcWMRTRcHGK-iA"

# Create the folders the rest of the notebook writes into.
for folder in [DATA_DIR, "outputs/figures", "outputs/results"]:
    os.makedirs(folder, exist_ok=True)

# Download only if the file is missing, so re-running this cell does not re-download it.
if not os.path.exists(DATA_PATH):
    gdown.download(id=DRIVE_FILE_ID, output=DATA_PATH, quiet=False)

print("train.csv present:", os.path.exists(DATA_PATH))

train.csv present: True


## 1. Load the data

**What:** reads `train.csv`, drops `Id`, converts `MSSubClass` to text, and separates the **features** (the inputs, `X`) from the **target** (the value we predict, `y`).

**Why:**
- `Id` is just a row number. Keeping it would let a model find accidental patterns in it.
- `MSSubClass` is stored as a number (20, 60, 190, ...), but each number is a code for a dwelling type. Type 60 is not "three times" type 20. Storing it as text makes the pipeline treat it as a **categorical** feature (one-hot encoded later) instead of a quantity.
- The target is `log(1 + SalePrice)`. In EDA, `SalePrice` was strongly right-skewed (skew 1.88); after the log it is almost symmetric (skew 0.12). Kaggle's metric is also RMSE on log price. Predictions are converted back to dollars later with `np.expm1`.

**What to look for:** shape `(1460, 80)` after dropping `Id`; `MSSubClass` dtype `object` (pandas' text type); 79 feature columns.

In [15]:
def load_data(path):
    """Load the Ames training data and apply the type decisions made in EDA.

    Input:  path to Kaggle's train.csv.
    Output: DataFrame without the Id column, with MSSubClass stored as text.
    """
    df = pd.read_csv(path)
    # Id is a row number with no information about the house.
    df = df.drop(columns="Id")
    # MSSubClass is a dwelling-type code, not a quantity, so treat it as categorical.
    df["MSSubClass"] = df["MSSubClass"].astype(str)
    return df


df = load_data(DATA_PATH)

# Features = every column except the price. Target = log price (skew 1.88 -> 0.12 in EDA).
X = df.drop(columns="SalePrice")
y_log = np.log1p(df["SalePrice"])

print("Data shape (rows, columns):", df.shape)
print("Number of features        :", X.shape[1])
print("MSSubClass dtype          :", df["MSSubClass"].dtype)
print("Skew of log price         :", round(y_log.skew(), 2))

Data shape (rows, columns): (1460, 80)
Number of features        : 79
MSSubClass dtype          : object
Skew of log price         : 0.12


**Result:** after dropping `Id`, the data has 1,460 rows and 80 columns (79 features plus `SalePrice`). `MSSubClass` is stored as text (`object`), so it will be one-hot encoded instead of treated as a number. The skew of the log target is 0.12, the same as in EDA, which confirms the target is close to symmetric.

## 2. Train / test split (80 / 20, stratified on price)

**What:** sets aside 20% of the houses as a **test set** that no step touches until the final evaluation. The other 80% is the **training set**, used for every later decision: imputation statistics, encodings, scaling, outlier removal, cross-validation and tuning.

**Why split first:** if any statistic (a median, an encoding, a scaler) were computed on all 1,460 houses, information about the test houses would leak into training. The test score would then look better than the model really is. This is called **data leakage**. Splitting before anything else prevents it.

**Why stratify:** a purely random split of 1,460 houses could, by chance, put too many expensive or cheap houses on one side. We cut log price into 10 equal-sized bins (**deciles**) and ask `train_test_split` to keep each bin's share the same in both sets. Both sets then cover the same price range.

*Note:* the bins use price only to decide which rows go where. Nothing learned from them is passed to a model, so this is not leakage.

**What to look for:** 1,168 training and 292 test houses; very similar price summaries (mean, quartiles, min, max) in the two columns.

In [16]:
TEST_SIZE = 0.2      # 80% train / 20% test
N_STRAT_BINS = 10    # deciles of log price: covers the price range, about 146 houses per bin

# Label each house with its price decile (0 = cheapest 10%, 9 = most expensive 10%).
price_bins = pd.qcut(y_log, q=N_STRAT_BINS, labels=False)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_log,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=price_bins,  # same share of each price decile in train and test
)

print("Training set:", X_train.shape)
print("Test set    :", X_test.shape)

# Compare the price distributions (in dollars) to confirm the stratification worked.
price_summary = pd.DataFrame({
    "train (USD)": np.expm1(y_train).describe(),
    "test (USD)": np.expm1(y_test).describe(),
}).round(0)
print("\nSalePrice summary by split:")
print(price_summary)

Training set: (1168, 79)
Test set    : (292, 79)

SalePrice summary by split:
       train (USD)  test (USD)
count       1168.0       292.0
mean      180977.0    180698.0
std        79285.0     80205.0
min        34900.0     35311.0
25%       129975.0    129750.0
50%       163000.0    163495.0
75%       213625.0    214875.0
max       755000.0    625000.0


**Result:** 1,168 houses are in the training set and 292 in the test set (80 / 20). The two price distributions are almost identical. The means are \$180,977 (train) and \$180,698 (test), and the medians are \$163,000 and \$163,495. All three quartiles are within \$1,250 of each other, so the stratification worked.

The maximum differs: \$755,000 in train and \$625,000 in test. Stratifying on deciles balances how many houses fall in each price band, not the single most extreme values. The top band is very wide, so one or two very expensive houses can land on either side. This means no test house costs more than the most expensive training house, which is fine for evaluation.

From here until the final evaluation, the test set is not used again.